# Phase 3: EDA

Exploring the 2016–2025 regular season. Reusable code lives in `src/nfl_ats/`. This notebook only loads, checks, and plots.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.ticker as mtick
import seaborn as sns

from nfl_ats.data import load_schedules, regular_season, validate_schedules, check_spread_sign
from nfl_ats.target import add_home_cover

sns.set_theme(style="whitegrid")
FIG_DIR = "../reports/figures"

In [ ]:
reg = regular_season(load_schedules())
validate_schedules(reg)
check_spread_sign(reg)

games = add_home_cover(reg)
graded = games.dropna(subset=["home_cover"])  # drop pushes (and any unplayed games)
graded.shape

## 1. Home cover rate by season

Error bars are bootstrap 95% CIs. With about 250 games a season, each bar spans about ±6 points.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.pointplot(data=graded, x="season", y="home_cover", errorbar=("ci", 95), n_boot=2000,
              seed=0, linestyle="none", capsize=0.3, ax=ax)
ax.axhline(0.5, color="gray", ls="--", lw=1, label="50% (coin flip)")
ax.axhline(0.524, color="crimson", ls=":", lw=1.5, label="52.4% (breakeven at -110)")
ax.axhline(graded["home_cover"].mean(), color="steelblue", lw=1, alpha=0.6,
           label=f"2016–2025 pooled: {graded['home_cover'].mean():.1%}")
ax.set(ylim=(0.3, 0.7), ylabel="Home cover rate (pushes excluded)", xlabel="Season",
       title="Home teams cover about half the time, every season")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/home_cover_by_season.png", dpi=150)

**Takeaway:** every season but one has an interval that includes 50%. The exception is 2019 (43%). With 10 seasons, one 95% interval missing 50% is about what chance alone produces. The closing line already prices in home field, so "always home" is not an edge.

## 2. Spread distribution and key numbers

Absolute values, so favorites and underdogs fold together. Red bars mark 3 and 7.

In [ ]:
played = games[games["margin"].notna()].copy()
played["abs_margin"] = played["margin"].abs()
played["abs_spread"] = played["spread_line"].abs()
KEY = {3, 7}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, col, title in [
    (axes[0], "abs_margin", "Final margin (absolute)"),
    (axes[1], "abs_spread", "Closing spread (absolute)"),
]:
    share = played[col].value_counts(normalize=True).sort_index().loc[lambda s: s.index <= 21]
    colors = ["crimson" if v in KEY else "steelblue" for v in share.index]
    width = 0.9 if col == "abs_margin" else 0.4  # spreads come in half points
    ax.bar(share.index, share.values, width=width, color=colors)
    ax.set(title=title, xlabel="Points", ylabel="Share of games", xticks=range(0, 22, 3))
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
fig.suptitle("Key numbers: margins pile up on 3 and 7, and the market sets lines around them")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/key_numbers.png", dpi=150)

In [ ]:
# Push rate on the most common whole-number spreads. Half-point spreads can't push.
whole = played[played["abs_spread"] % 1 == 0]
(whole.groupby("abs_spread")
      .agg(games=("game_id", "size"), push_rate=("is_push", "mean"))
      .query("games >= 50")
      .sort_values("push_rate", ascending=False)
      .round(3))

**Takeaway:** 14% of games end with a margin of exactly 3, and 8.5% with exactly 7. The market knows it: 3 is the most common closing spread, flanked by 2.5 and 3.5. A spread of exactly 3 pushes about 10% of the time, while 1 and 5 almost never do. This means a half point across 3 or 7 is worth far more than one across 5 or 9, which a smooth margin model ignores (Phase 1).

## 3. Margin vs. spread residuals

`resid = margin - spread_line`. A positive value means the home team beat the line. If the closing line is unbiased, residuals center on 0 and the binned means sit on y = x.

In [ ]:
played["resid"] = played["margin"] - played["spread_line"]  # > 0 means the home team beat the line
sd = played["resid"].std()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: how far games land from the line, against a normal curve with the same mean and SD.
sns.histplot(played["resid"], binwidth=1, stat="density", color="steelblue", ax=axes[0])
x = np.linspace(-45, 45, 400)
axes[0].plot(x, np.exp(-0.5 * ((x - played["resid"].mean()) / sd) ** 2) / (sd * np.sqrt(2 * np.pi)),
             color="crimson", lw=1.5, label=f"Normal, SD = {sd:.1f}")
axes[0].set(xlim=(-45, 45), xlabel="Margin minus spread (points)",
            title=f"Residuals: mean {played['resid'].mean():+.2f}, SD {sd:.1f}")
axes[0].legend(fontsize=8)

# Right: mean margin within spread bins. If the line is unbiased, the points sit on y = x.
sns.regplot(data=played, x="spread_line", y="margin", x_bins=np.arange(-14, 15, 2),
            x_ci=95, n_boot=2000, seed=0, truncate=True, color="steelblue", ax=axes[1])
axes[1].plot([-15, 15], [-15, 15], color="gray", ls="--", lw=1, label="y = x (line is unbiased)")
axes[1].set(xlabel="Closing spread (home expected margin)", ylabel="Actual home margin",
            title="Binned mean margin vs. spread")
axes[1].legend(fontsize=8)

fig.tight_layout()
fig.savefig(f"{FIG_DIR}/margin_vs_spread.png", dpi=150)

In [ ]:
# Cover rate by spread bucket. Each bucket has 160-690 games, so the CIs are ±4 to ±8 points.
buckets = pd.cut(played["spread_line"], [-30, -7.5, -3.5, 0, 3.5, 7.5, 30])
played.groupby(buckets, observed=True).agg(
    games=("resid", "size"),
    mean_resid=("resid", "mean"),
    home_cover=("home_cover", "mean"),
).round(3)

**Takeaway:** the closing line is unbiased. Mean residual is +0.04 points, and margin regressed on spread has a slope of 1.04 and an intercept of -0.03. The spread alone explains only about 20% of the variance in margin (R² = 0.20), and the residual SD is **12.7 points**: a typical game misses the line by almost two touchdowns. The spike at 0 comes from pushes and key numbers. No spread bucket's cover rate is clearly away from 50%. Home dogs of 7.5+ cover 56%, but n = 164, so the 95% CI is about ±7.6 points. That's noise, not an angle.

## 4. Rest-day differences vs. cover rate

`rest_diff = home_rest - away_rest`. Most games are 0 (both teams on a normal week).

In [ ]:
# Rest advantage for the home team. ±7 is mostly one team coming off a bye; ±3 is Thursday games.
played["rest_diff"] = played["home_rest"] - played["away_rest"]
played["rest_bucket"] = pd.cut(played["rest_diff"], [-99, -4, -1, 0, 3, 99],
                               labels=["≤ -4", "-3 to -1", "0", "+1 to +3", "≥ +4"])
played["rest_bucket"].value_counts(sort=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: does rest move the result, and does the line move with it?
by_rest = (played.groupby("rest_bucket", observed=True)[["margin", "spread_line"]].mean()
                 .rename(columns={"margin": "Actual home margin", "spread_line": "Closing spread"})
                 .reset_index()
                 .melt(id_vars="rest_bucket", var_name="measure", value_name="points"))
sns.pointplot(data=by_rest, x="rest_bucket", y="points", hue="measure", ax=axes[0])
axes[0].set(xlabel="Home rest advantage (days)", ylabel="Points (home perspective)",
            title="The line adds about 2 points for a big home rest edge")
axes[0].legend(title=None, fontsize=8)

# Right: once the line has priced rest in, is anything left over?
graded_rest = played.dropna(subset=["home_cover"])
sns.pointplot(data=graded_rest, x="rest_bucket", y="home_cover", errorbar=("ci", 95), n_boot=2000,
              seed=0, linestyle="none", capsize=0.3, ax=axes[1])
axes[1].axhline(0.5, color="gray", ls="--", lw=1)
axes[1].axhline(0.524, color="crimson", ls=":", lw=1.5, label="52.4% breakeven")
axes[1].set(ylim=(0.3, 0.7), xlabel="Home rest advantage (days)", ylabel="Home cover rate",
            title="Cover rate stays flat: no rest edge left over")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].legend(fontsize=8)

fig.tight_layout()
fig.savefig(f"{FIG_DIR}/rest_vs_cover.png", dpi=150)

**Takeaway:** the market prices rest. The closing spread rises from +1.3 when the home team is at a 4+ day rest disadvantage to +3.4 when it has a 4+ day advantage, mostly from byes. Actual margins roughly follow, but with about 140 games per bucket and a 12.7-point SD, those margin means are noisy (SE ≈ 1 point). The cover rate is flat at about 48–50% in every bucket, and all CIs include 50%. Raw rest_diff won't add signal beyond the spread on its own, but it's cheap and may interact with team strength. It's in v1 anyway, per the design (decision #15): the backtest decides whether it earns its place.